## Cell 1: Install & Import Transformer Dependencies

In [1]:
import os
import torch
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

## Cell 2: Load Data Splits & Initialize Tokenizer

In [2]:
splits_dir = "../data/processed/splits/"

train_df = pd.read_csv(os.path.join(splits_dir, "train_split.csv"))
val_df = pd.read_csv(os.path.join(splits_dir, "val_split.csv"))
test_df = pd.read_csv(os.path.join(splits_dir, "test_split.csv"))

label_encoder = joblib.load(os.path.join(splits_dir, "label_encoder.pkl"))
class_names = list(label_encoder.classes_)
num_labels = len(class_names)

# Load XLM-RoBERTa Base Tokenizer
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    # Tokenizes English text (or concatenate cleaned_amharic if evaluating multilingual inputs)
    return tokenizer(examples["cleaned_english"], truncation=True, max_length=128, padding="max_length")

# Convert pandas DataFrames to Hugging Face Datasets
train_hf = Dataset.from_pandas(train_df[["cleaned_english", "target"]].rename(columns={"target": "label"}))
val_hf = Dataset.from_pandas(val_df[["cleaned_english", "target"]].rename(columns={"target": "label"}))
test_hf = Dataset.from_pandas(test_df[["cleaned_english", "target"]].rename(columns={"target": "label"}))

# Apply Tokenization
train_dataset = train_hf.map(tokenize_function, batched=True)
val_dataset = val_hf.map(tokenize_function, batched=True)
test_dataset = test_hf.map(tokenize_function, batched=True)

Map:   0%|          | 0/20996 [00:00<?, ? examples/s]

Map:   0%|          | 0/2624 [00:00<?, ? examples/s]

Map:   0%|          | 0/2625 [00:00<?, ? examples/s]

## Cell 3: Compute Class Weights & Define Evaluation Metrics

In [3]:
from sklearn.utils.class_weight import compute_class_weight

# Calculate weights to handle class imbalance in Cross-Entropy Loss
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["target"]),
    y=train_df["target"]
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float)

# Custom Trainer to pass Weighted Loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights_tensor.to(model.device))
        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    
    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds, average="macro")
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    
    return {
        "accuracy": round(acc, 4),
        "macro_recall": round(r_macro, 4),
        "macro_f1": round(f1_macro, 4),
        "weighted_f1": round(f1_weighted, 4)
    }

## Cell 4: Train XLM-RoBERTa

In [5]:
# Initialize Pre-trained XLM-RoBERTa Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir="../saved_models/xlm_roberta_checkpoints",
    eval_strategy="epoch",          # Updated from evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=100,
    fp16=torch.cuda.is_available()  # Uses GPU if available
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,     # Updated from tokenizer
    compute_metrics=compute_metrics
)

# Start Fine-Tuning
trainer.train()

# Save Final Fine-Tuned Weights
model.save_pretrained("../saved_models/xlm_roberta_mental_health")
tokenizer.save_pretrained("../saved_models/xlm_roberta_mental_health")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\ProgramData\conda_envs\mental_ai\Lib\site-packages\torch\utils\data\dataloader.py:759: User

Epoch,Training Loss,Validation Loss,Accuracy,Macro Recall,Macro F1,Weighted F1
1,0.722551,0.625018,0.812100,0.790700,0.801300,0.812900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\ProgramData\conda_envs\mental_ai\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

## Cell 5: Evaluate XLM-RoBERTa on Test Set & Final Model Selection Table

In [ ]:
# Predict on Test Set
test_results = trainer.predict(test_dataset)
test_preds = np.argmax(test_results.predictions, axis=1)

# Generate Metrics
xlm_metrics = compute_metrics((test_results.predictions, test_results.label_ids))

# Final Comparison Table across ALL 4 models
final_comparison = pd.DataFrame([
    {"Model": "XLM-RoBERTa (Fine-Tuned)", "Accuracy": xlm_metrics["accuracy"], "Macro Recall": xlm_metrics["macro_recall"], "Macro F1-Score": xlm_metrics["macro_f1"], "Weighted F1-Score": xlm_metrics["weighted_f1"]},
    {"Model": "Logistic Regression", "Accuracy": 0.7592, "Macro Recall": 0.7471, "Macro F1-Score": 0.7491, "Weighted F1-Score": 0.7586},
    {"Model": "XGBoost Classifier", "Accuracy": 0.7459, "Macro Recall": 0.7351, "Macro F1-Score": 0.7431, "Weighted F1-Score": 0.7462},
    {"Model": "Multinomial Naive Bayes", "Accuracy": 0.6674, "Macro Recall": 0.5978, "Macro F1-Score": 0.6123, "Weighted F1-Score": 0.6462}
]).sort_values(by="Macro F1-Score", ascending=False)

print("========== FINAL MULTI-MODEL BENCHMARK ==========")
final_comparison